In [21]:
from bs4 import BeautifulSoup
import requests
import os
import csv
from urllib.parse import urljoin
from datetime import datetime

url_base = "https://en.wikipedia.org/wiki/Andrey_Kolmogorov"


In [22]:
def coletar_infos(url_base = url_base):
    """
    Função coleta dados de paginas da wikipedia, devolve titulo,  primeiro paragrafo, lista de links de imagens, lista de links de sites de wikipedia
    """

    headers = {"User-Agent": "Grupo do Balacobaco"}

    response = requests.get(url_base, headers=headers)
    response.encoding = 'utf-8'
    soup = BeautifulSoup(response.text, "html.parser")

    # 1. Título
    titulo = soup.find("h1", id="firstHeading").get_text()
    
    print(titulo)


    # 2. Primeiro paragrafo
    content_div = soup.find("div", id="bodyContent")
    p_paragrafo = "Resumo não encontrado."
    
    if content_div:
        # Busca todos os parágrafos que são filhos diretos ou descendentes
        todos_p = content_div.find_all("p")
        for p in todos_p:
            # Verifica se o parágrafo tem texto útil (ignora vazios ou só espaços)
            p_paragrafo = p.get_text().strip()
            if len(p_paragrafo) > 10: # Um parágrafo real costuma ter mais que 20 caracteres
                break # Encontrou o primeiro parágrafo válido, para o loop


    # 3) Encontrar links para imagens
    # Busca todas as tags de imagem
    imagens = soup.find_all("img")

    print(f"Total de imagens encontradas: {len(imagens)}")

    img_links = []

    for img in imagens:
        # Src guarda as urls relativas
        src = img.get('src')
        if src:
            # o Urljoin funciona transformado urls relativas em absolutas com base na url base
            link_completo = urljoin(url_base, src)
            img_links.append(link_completo)


    # 4 Encontrar todos os links wiki 
    # Um set ára não ter links duplicados
    wiki_links = set()

    links = soup.find_all("a", href=True)

    for link in links:
        href = link.get("href")

        # Filtro:
        # 1. Deve começar com /wiki/
        # 2. Não deve conter ":" (para evitar páginas de Categoria, Ajuda, Ficheiro, etc.)
        if href.startswith("/wiki/") and ":" not in href:
            link_completo = urljoin(url_base, href)
            if link_completo == url_base:
                continue
            wiki_links.add(link_completo)
    print(f"Total de links encontrados: {len(wiki_links)}")
            
    return response, titulo, p_paragrafo, img_links, wiki_links

In [23]:
# Coletando da url base
response, titulo, p_paragrafo, img_links, wiki_links = coletar_infos()

Andrey Kolmogorov
Total de imagens encontradas: 14
Total de links encontrados: 254


In [24]:
def salvar_dados_wiki(html_content, lista_imagens, titulo, primeiro_paragrafo, lista_links):
    """
    Salva HTML do site, em uma pasta "html". 
    Salva os dados da pagina em csvs/(titulo da pagina)/imagescsv e /arquivo.csv, o primeiro com link para as imagens, o segundo com os dados da pagina
    """
    
    
    diretorio_csv = f"csvs/{titulo}"
    diretorio_html = "html"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    os.makedirs(diretorio_csv, exist_ok=True)
    os.makedirs(diretorio_html, exist_ok=True)
    
    # 1. Salvar HTML
    with open(os.path.join(diretorio_html, f"{titulo}.html"), "w", encoding="utf-8") as f:
        f.write(html_content)
    
    # 2. Salvar Imagens
    with open(os.path.join(diretorio_csv, "images.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["link_imagem", "data_coleta"])
        for img in lista_imagens:
            writer.writerow([img, timestamp])
            
    # 3. Salvar Conteúdo
    with open(os.path.join(diretorio_csv, "arquivo.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["titulo", "primeiro_paragrafo", "links_artigos", "data_coleta"])
        writer.writerow([titulo, primeiro_paragrafo, "|".join(lista_links), timestamp])

    print(f"✅ Dados de '{titulo}' salvos com sucesso!")

In [25]:
salvar_dados_wiki(
    html_content=response.text
    lista_imagens=img_links,
    titulo=titulo,
    primeiro_paragrafo=p_paragrafo,
    lista_links=list(wiki_links)
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (650934446.py, line 2)

In [ ]:
lista = list(wiki_links)
for link in lista:
    response, titulo, p_paragrafo, img_links, wiki_links = coletar_infos(link)
    salvar_dados_wiki(
        html_content=response.text,
        lista_imagens=img_links,
        titulo=titulo,
        primeiro_paragrafo=p_paragrafo,
        lista_links=list(wiki_links)
    )

Novodevichy Cemetery
Total de imagens encontradas: 27
Total de links encontrados: 47
✅ Dados de 'Novodevichy Cemetery' salvos com sucesso!
Kolmogorov–Zurbenko filter
Total de imagens encontradas: 35
Total de links encontrados: 56
✅ Dados de 'Kolmogorov–Zurbenko filter' salvos com sucesso!
Ingrid Daubechies
Total de imagens encontradas: 10
Total de links encontrados: 434
✅ Dados de 'Ingrid Daubechies' salvos com sucesso!
Lars Ahlfors
Total de imagens encontradas: 15
Total de links encontrados: 231
✅ Dados de 'Lars Ahlfors' salvos com sucesso!
Quasi-arithmetic mean
Total de imagens encontradas: 95
Total de links encontrados: 42
✅ Dados de 'Quasi-arithmetic mean' salvos com sucesso!
Lobachevsky Prize
Total de imagens encontradas: 6
Total de links encontrados: 66
✅ Dados de 'Lobachevsky Prize' salvos com sucesso!
Valery Vasilevich Kozlov
Total de imagens encontradas: 11
Total de links encontrados: 18
✅ Dados de 'Valery Vasilevich Kozlov' salvos com sucesso!
Land tenure
Total de imagens enc